In [1]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

text = "I love AI"

inputs = tokenizer(text, return_tensors="pt", padding=True)
translated = model.generate(**inputs)

output = tokenizer.decode(translated[0], skip_special_tokens=True)

print(output)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Anh yêu em


In [2]:
from datasets import load_dataset

dataset = load_dataset("text", data_files="data.txt")

In [3]:
def preprocess(example):
    parts = example["text"].split("\t")
    
    if len(parts) < 2:
        return {"src": "", "tgt": ""}  # bỏ dòng lỗi
    
    return {
        "src": parts[0],
        "tgt": parts[1]
    }
dataset = dataset.map(preprocess)

In [4]:
def tokenize(example):
    inputs = tokenizer(example["src"], truncation=True, padding="max_length")
    targets = tokenizer(example["tgt"], truncation=True, padding="max_length")

    inputs["labels"] = targets["input_ids"]
    return inputs

dataset = dataset.map(tokenize)

In [5]:
import sys
print(sys.executable)

D:\jupiter\python.exe


In [6]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./model",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=10,
    predict_with_generate=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"]
)

trainer.train()

C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=2.95915158589681, metrics={'train_runtime': 62.4138, 'train_samples_per_second': 0.144, 'train_steps_per_second': 0.048, 'total_flos': 1220341137408.0, 'train_loss': 2.95915158589681, 'epoch': 3.0})

In [7]:
trainer.save_model("my_translator")
tokenizer.save_pretrained("my_translator")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_translator\\tokenizer_config.json',
 'my_translator\\vocab.json',
 'my_translator\\source.spm',
 'my_translator\\target.spm',
 'my_translator\\added_tokens.json')

In [8]:
from transformers import MarianMTModel, MarianTokenizer

model = MarianMTModel.from_pretrained("my_translator")
tokenizer = MarianTokenizer.from_pretrained("my_translator")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [9]:
import os

print(os.listdir())
print(os.path.exists("my_translator"))

['.anaconda', '.android', '.cache', '.conda', '.copilot', '.dbus-keyrings', '.dotnet', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '.Ld9VirtualBox', '.matplotlib', '.mitmproxy', '.nuget', '.opengenius', '.oracle_jre_usage', '.p2', '.packettracer', '.python_history', '.templateengine', '.thinkbuzan', '.thumbnails', '.vscode', '3D Objects', 'AppData', 'Application Data', 'Autodesk', 'bangdiemsv.ipynb', 'BART.ipynb', 'bth9.ipynb', 'bul.txt', 'Cisco Packet Tracer 8.1.1', 'Cisco Packet Tracer 8.2.2', 'Contacts', 'Cookies', 'd4ac4633ebd6440fa397b84f1bc94a3c.7z', 'data', 'data.txt', 'Demo_MASS.ipynb', 'demo_mohinh_part2.ipynb', 'Demo_mT5.ipynb', 'DEMO_T5.ipynb', 'Desktop', 'Documents', 'Downloads', 'DSK43.xlsx', 'dsnv3.csv', 'du_an_NLP.ipynb', 'Favorites', 'GPT_model.ipynb', 'inittk.ini', 'inst.ini', 'IntelGraphicsProfiles', 'khach_hang.ipynb', 'Links', 'Local Settings', 'LTHDT_SINHVIEN.ipynb', 'machine-translation-seq2seq-lstms.ipynb', 'MarianMT_.ipynb', 'mentalment

In [12]:
pip install gradio

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [16]:
from transformers import MarianMTModel, MarianTokenizer
import gradio as gr

model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def translate(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

gr.Interface(
    fn=translate,
    inputs="text",
    outputs="text",
    title="English → Vietnamese Translator"
).launch()

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
